In [1]:
import subprocess, sys
for pkg in ["python-pptx", "openai", "pandas", "ipywidgets", "matplotlib"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {pkg}")
print("\n🎉 Ready!")

✅ python-pptx
✅ openai
✅ pandas
✅ ipywidgets
✅ matplotlib

🎉 Ready!


In [2]:
import io, json, re, base64, time, textwrap
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless — no display needed
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from openai import OpenAI

from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_AUTO_SIZE
from pptx.chart.data import ChartData
from pptx.enum.chart import XL_CHART_TYPE

print("✅ All imports ready")

✅ All imports ready


In [3]:
# ── Model ─────────────────────────────────────────────────────────────────────
BASE_URL    = "http://localhost:8000/v1"
API_KEY     = "abc-123"
MODEL       = "Qwen2.5-7B"
TEMPERATURE = 0.0        # deterministic — critical for JSON reliability
MAX_TOKENS  = 4000  # increased for richer content generation
MAX_RETRIES = 2

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_PPT  = "Executive_Intelligence_Report.pptx"
CHARTS_DIR  = Path("_charts")   # temp folder for chart images
CHARTS_DIR.mkdir(exist_ok=True)

# ── Palette — Charcoal Minimal + electric teal accent ─────────────────────────
# Dark backgrounds for title/section dividers; white for content slides
C_CHARCOAL  = RGBColor(0x1C, 0x1C, 0x2E)  # near-black — title bg
C_NAVY      = RGBColor(0x0D, 0x1B, 0x4B)  # deep navy — divider slides
C_TEAL      = RGBColor(0x00, 0xC2, 0xB2)  # electric teal — accent
C_GOLD      = RGBColor(0xF5, 0xA6, 0x23)  # amber gold — KPI highlight
C_RED_SOFT  = RGBColor(0xE5, 0x3E, 0x3E)  # soft red — risk
C_GREEN     = RGBColor(0x1A, 0xB3, 0x73)  # green — opportunity
C_WHITE     = RGBColor(0xFF, 0xFF, 0xFF)
C_OFFWHITE  = RGBColor(0xF7, 0xF8, 0xFC)  # content slide bg
C_MUTED     = RGBColor(0x6B, 0x7A, 0x99)  # caption / label text
C_DARK_TEXT = RGBColor(0x1C, 0x1C, 0x2E)  # body text on light bg

SLIDE_W = Inches(13.33)
SLIDE_H = Inches(7.5)

# ── Connect ───────────────────────────────────────────────────────────────────
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("Testing connection...")
try:
    ids = [m.id for m in client.models.list().data]
    print(f"✅ Connected — models: {ids}")
    if MODEL not in ids:
        print(f"⚠️  '{MODEL}' not found. Check --served-model-name.")
except Exception as e:
    print(f"❌ {e}")

Task was destroyed but it is pending!
task: <Task pending name='Task-95' coro=<_async_in_context.<locals>.run_in_context() done, defined at /usr/local/lib/python3.12/dist-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-96' coro=<Kernel.shell_main() running at /usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /usr/local/lib/python3.12/dist-packages/zmq/eventloop/zmqstream.py:563]>
/usr/lib/python3.12/collections/__init__.py:449: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  result = tuple_new(cls, iterable)
Task was destroyed but it is pending!
task: <Task pending name='Task-96' coro=<Kernel.shell_main() running at /usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


Testing connection...
✅ Connected — models: ['Qwen2.5-7B']


In [4]:
uploaded_data = {"content": None, "filename": None, "filetype": None}

uploader   = widgets.FileUpload(accept=".csv,.txt", multiple=False,
                                description="Upload File", button_style="primary",
                                layout=widgets.Layout(width="200px"))
status_out = widgets.Output()

def on_upload(change):
    with status_out:
        clear_output(wait=True)
        if not uploader.value: return
        fi    = uploader.value[0]
        fname = fi["name"]
        raw   = bytes(fi["content"])   # bytes() fixes memoryview bug
        ext   = Path(fname).suffix.lower()
        uploaded_data.update({"filename": fname, "filetype": ext})
        if ext == ".txt":
            uploaded_data["content"] = raw.decode("utf-8", errors="replace")
            print(f"✅ TXT: {fname}  ({len(uploaded_data['content']):,} chars)")
        elif ext == ".csv":
            df = pd.read_csv(io.BytesIO(raw))
            uploaded_data["content"] = df
            print(f"✅ CSV: {fname}  ({len(df):,} rows × {len(df.columns)} cols)")
            display(df.head(3))
        else:
            print(f"❌ Unsupported: {ext}")

uploader.observe(on_upload, names="value")
display(widgets.VBox([
    widgets.HTML("<b>📎 Upload .csv or .txt file:</b>"),
    uploader, status_out
]))

In [5]:
def extract_kpis(df: pd.DataFrame) -> list[dict]:
    """
    Automatically surface the most interesting numbers from a DataFrame.
    Returns a list of {label, value, delta, sentiment} dicts for KPI cards.
    """
    kpis = []
    num_cols = df.select_dtypes(include="number").columns.tolist()

    for col in num_cols[:8]:   # cap at 8 columns to scan
        series = df[col].dropna()
        if len(series) < 2: continue

        total   = series.sum()
        mean    = series.mean()
        mx      = series.max()
        mn      = series.min()

        # Compute simple trend: compare first half vs second half
        mid     = len(series) // 2
        first_h = series.iloc[:mid].mean() if mid > 0 else mean
        second_h= series.iloc[mid:].mean()
        pct_chg = ((second_h - first_h) / first_h * 100) if first_h != 0 else 0

        label = col.replace("_", " ").title()

        # Format value nicely
        if total > 1_000_000:
            val_str = f"${total/1_000_000:.1f}M" if "revenue" in col.lower() or "sales" in col.lower() else f"{total/1_000_000:.1f}M"
        elif total > 1_000:
            val_str = f"{total:,.0f}"
        else:
            val_str = f"{mean:.1f} avg"

        delta_str = f"{pct_chg:+.1f}%" if abs(pct_chg) > 1 else "Stable"
        # Sentiment thresholds: >+1% = positive, <-1% = negative, else neutral
        if pct_chg > 1:
            sentiment = "positive"
        elif pct_chg < -1:
            sentiment = "negative"
        else:
            sentiment = "neutral"

        kpis.append({"label": label, "value": val_str,
                     "delta": delta_str, "sentiment": sentiment})

    return kpis[:6]   # max 6 KPI cards on a slide


def process_txt(text: str, max_chars: int = 8000) -> str:
    text = text.strip()
    return text if len(text) <= max_chars else text[:max_chars] + "\n[...truncated]"


def process_csv(df: pd.DataFrame) -> str:
    lines = ["=== DATASET OVERVIEW ===",
             f"Shape: {len(df):,} rows × {len(df.columns)} columns",
             f"Columns: {', '.join(df.columns)}", ""]

    num_df = df.select_dtypes(include="number")
    if not num_df.empty:
        lines += ["=== NUMERIC STATISTICS ===", num_df.describe().round(2).to_string(), ""]
        for col in num_df.columns:
            pc = num_df[col].pct_change().mean()
            if abs(pc) > 0.03:
                lines.append(f"  Trend: {col} avg {'+' if pc>0 else ''}{pc*100:.1f}% per period")
        lines.append("")

    cat_df = df.select_dtypes(include=["object","category"])
    if not cat_df.empty:
        lines.append("=== CATEGORICAL BREAKDOWN ===")
        for col in cat_df.columns[:6]:
            top = df[col].value_counts().head(5)
            lines.append(f"  {col}: " + ", ".join(f"{v}({c})" for v,c in top.items()))
        lines.append("")

    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        lines.append("=== MISSING DATA ===")
        for col, cnt in missing.items():
            lines.append(f"  {col}: {cnt} missing ({cnt/len(df)*100:.1f}%)")
        lines.append("")

    lines += ["=== SAMPLE (first 5 rows) ===", df.head(5).to_string(index=False)]
    return "\n".join(lines)




def calculate_health_score(kpis: list, insights: dict) -> int:
    """
    Programmatically compute a business health score (0-100).

    Scoring logic (deterministic, reproducible):
      - KPI sentiment component  (0-50 pts):
          Each KPI is worth up to 50/n points.
          positive  -> full points
          neutral   -> half points
          negative  -> zero points
      - Insight quality component (0-30 pts):
          +10 if >= 3 non-placeholder opportunities exist
          +10 if <= 2 risk items (fewer serious risks = healthier)
          +10 if >= 3 key findings (rich data = confident score)
      - Data coverage component (0-20 pts):
          +5 per non-empty insight list (max 4 lists x 5 pts)

    Returns an integer in [0, 100].
    """
    score = 0

    # ── KPI component (max 50) ──────────────────────────────────────────────
    if kpis:
        pts_each = 50 / len(kpis)
        for k in kpis:
            s = str(k.get("sentiment", "neutral")).lower()
            if s == "positive":
                score += pts_each
            elif s == "neutral":
                score += pts_each * 0.5
            # negative -> 0
    else:
        score += 25   # no KPI data — neutral baseline

    # ── Insight quality component (max 30) ──────────────────────────────────
    def _non_empty(lst):
        return [x for x in (lst or []) if x and str(x).strip()
                and not any(p in str(x).lower() for p in
                            {"not specified","not available","n/a","none","not provided"})]

    opps  = _non_empty(insights.get("opportunities",  []))
    risks = _non_empty(insights.get("risks",          []))
    finds = _non_empty(insights.get("key_findings",   []))

    if len(opps)  >= 3: score += 10
    if len(risks) <= 2: score += 10   # fewer material risks = healthier
    if len(finds) >= 3: score += 10

    # ── Data coverage component (max 20) ────────────────────────────────────
    for key in ("key_findings", "trends", "risks", "opportunities"):
        lst = insights.get(key, [])
        if isinstance(lst, list) and len([x for x in lst if x]) >= 1:
            score += 5

    return max(0, min(100, round(score)))


# ── Run ───────────────────────────────────────────────────────────────────────
if uploaded_data["content"] is None:
    print("⚠️  Upload a file in Cell 4 first.")
else:
    ft = uploaded_data["filetype"]
    if ft == ".csv":
        data_summary = process_csv(uploaded_data["content"])
        kpis         = extract_kpis(uploaded_data["content"])
        print(f"✅ CSV processed — {len(data_summary):,} chars, {len(kpis)} KPIs extracted")
        for k in kpis:
            print(f"   📊 {k['label']}: {k['value']}  ({k['delta']})")
    else:
        data_summary = process_txt(uploaded_data["content"])
        kpis         = []
        print(f"✅ TXT processed — {len(data_summary):,} chars")

✅ CSV processed — 1,432 chars, 4 KPIs extracted
   📊 Units Sold: 16,121  (+30.8%)
   📊 Revenue: $14.4M  (+34.0%)
   📊 Target: 17,120  (+25.3%)
   📊 Customer Satisfaction: 4.5 avg  (+7.6%)


In [6]:
# Charcoal/teal style for all charts
CHART_BG    = "#F7F8FC"
CHART_TEAL  = "#00C2B2"
CHART_NAVY  = "#0D1B4B"
CHART_GOLD  = "#F5A623"
CHART_RED   = "#E53E3E"
CHART_GREEN = "#1AB373"
CHART_MUTED = "#6B7A99"
PALETTE     = [CHART_TEAL, CHART_NAVY, CHART_GOLD, CHART_GREEN, CHART_RED,
               "#7B61FF", "#FF6B35", "#00B4D8"]

def _style_ax(ax, title=""):
    """Apply consistent styling to a matplotlib axis."""
    ax.set_facecolor(CHART_BG)
    ax.tick_params(colors=CHART_MUTED, labelsize=9)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.yaxis.grid(True, color="#E0E0E0", linewidth=0.7, linestyle="--")
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, fontsize=11, fontweight="bold",
                     color=CHART_NAVY, pad=10, loc="left")


def make_trend_chart(df: pd.DataFrame, path: str) -> bool:
    """Line chart: first numeric col over the first text/date col (or index)."""
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if not num_cols: return False
    cat_cols = df.select_dtypes(include=["object","category"]).columns.tolist()

    # Use up to 3 numeric series for comparison richness
    series_cols = num_cols[:3]
    x_labels = df[cat_cols[0]].astype(str) if cat_cols else df.index.astype(str)

    # If too many rows, aggregate or sample
    if len(df) > 24:
        step = max(1, len(df)//24)
        df   = df.iloc[::step]
        x_labels = df[cat_cols[0]].astype(str) if cat_cols else df.index.astype(str)

    fig, ax = plt.subplots(figsize=(9, 3.8), facecolor=CHART_BG)
    for i, col in enumerate(series_cols):
        vals = pd.to_numeric(df[col], errors="coerce").fillna(0).values
        ax.plot(range(len(vals)), vals, color=PALETTE[i], linewidth=2.2,
                marker="o", markersize=4, label=col.replace("_"," ").title())
        # Fill under primary line
        if i == 0:
            ax.fill_between(range(len(vals)), vals, alpha=0.08, color=PALETTE[0])

    tick_step = max(1, len(x_labels)//8)
    ax.set_xticks(range(0, len(x_labels), tick_step))
    ax.set_xticklabels(list(x_labels)[::tick_step], rotation=30, ha="right", fontsize=8)
    if len(series_cols) > 1:
        ax.legend(fontsize=8, framealpha=0.5, loc="upper left")
    _style_ax(ax, "Performance Trend")
    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=CHART_BG)
    plt.close()
    return True


def make_category_chart(df: pd.DataFrame, path: str) -> bool:
    """Horizontal bar chart for first categorical × first numeric."""
    cat_cols = df.select_dtypes(include=["object","category"]).columns.tolist()
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if not cat_cols or not num_cols: return False

    grp = df.groupby(cat_cols[0])[num_cols[0]].sum().nlargest(8).sort_values()
    if grp.empty: return False

    fig, ax = plt.subplots(figsize=(9, 3.8), facecolor=CHART_BG)
    colors  = [CHART_TEAL if i == len(grp)-1 else CHART_NAVY for i in range(len(grp))]
    bars    = ax.barh(grp.index.astype(str), grp.values, color=colors,
                      height=0.55, edgecolor="none")

    # Value labels on bars
    for bar, val in zip(bars, grp.values):
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                f"{val:,.0f}", va="center", fontsize=8, color=CHART_MUTED)

    title = f"{num_cols[0].replace('_',' ').title()} by {cat_cols[0].replace('_',' ').title()}"
    _style_ax(ax, title)
    ax.xaxis.grid(False)
    ax.yaxis.grid(False)
    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=CHART_BG)
    plt.close()
    return True


def make_distribution_chart(df: pd.DataFrame, path: str) -> bool:
    """Stacked bar or grouped bar showing second categorical breakdown."""
    cat_cols = df.select_dtypes(include=["object","category"]).columns.tolist()
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if len(cat_cols) < 2 or not num_cols:
        # Fall back: histogram of primary numeric
        if not num_cols: return False
        fig, ax = plt.subplots(figsize=(9, 3.8), facecolor=CHART_BG)
        vals = df[num_cols[0]].dropna()
        ax.hist(vals, bins=12, color=CHART_TEAL, edgecolor="none", alpha=0.85)
        _style_ax(ax, f"Distribution — {num_cols[0].replace('_',' ').title()}")
        plt.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=CHART_BG)
        plt.close()
        return True

    pivot = df.groupby([cat_cols[0], cat_cols[1]])[num_cols[0]].sum().unstack(fill_value=0)
    if pivot.empty or len(pivot.columns) > 8: return False

    fig, ax = plt.subplots(figsize=(9, 3.8), facecolor=CHART_BG)
    pivot.plot(kind="bar", ax=ax, color=PALETTE[:len(pivot.columns)],
               edgecolor="none", width=0.65)
    ax.legend(fontsize=8, framealpha=0.5, loc="upper right")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    title = f"{num_cols[0].replace('_',' ').title()} by {cat_cols[0].title()} & {cat_cols[1].title()}"
    _style_ax(ax, title)
    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=CHART_BG)
    plt.close()
    return True


# ── Run chart generation ───────────────────────────────────────────────────────
chart_paths = {}   # {"trend": path, "category": path, "distribution": path}

if uploaded_data["filetype"] == ".csv" and uploaded_data["content"] is not None:
    df_charts = uploaded_data["content"]

    tasks = [
        ("trend",        make_trend_chart,        str(CHARTS_DIR/"trend.png")),
        ("category",     make_category_chart,     str(CHARTS_DIR/"category.png")),
        ("distribution", make_distribution_chart, str(CHARTS_DIR/"distribution.png")),
    ]
    for name, fn, p in tasks:
        try:
            ok = fn(df_charts, p)
            if ok:
                chart_paths[name] = p
                print(f"✅ Chart '{name}' generated → {p}")
            else:
                print(f"⏭️  Chart '{name}' skipped (insufficient columns)")
        except Exception as e:
            print(f"⚠️  Chart '{name}' failed: {e}")
else:
    print("ℹ️  TXT file — chart generation skipped")

✅ Chart 'trend' generated → _charts/trend.png
✅ Chart 'category' generated → _charts/category.png
✅ Chart 'distribution' generated → _charts/distribution.png


In [7]:
# ════════════════════════════════════════════════════════════════════════════════
# PROMPTS — Consulting-Grade Executive Language
# ════════════════════════════════════════════════════════════════════════════════

STEP1_SYSTEM = """You are a senior management consultant at a top-tier strategy firm.
You write board-ready executive briefings: direct, evidence-based, commercially sharp.
Every bullet must contain an observation AND its business significance.
Never write single-line labels. Never use filler language.
Think: what would a McKinsey partner say in the first 10 minutes of a C-suite debrief?"""

STEP1_USER = """Analyse the data below and produce a rich executive briefing.

FORMATTING RULES (READ CAREFULLY):
- Each bullet = 1 complete thought: observation + significance + implication.
- Target 20–35 words per bullet. Write 2-sentence bullets where needed.
- Use numbers and metrics from the data wherever possible.
- Never write placeholders like "N/A", "Not specified", or vague labels.
- Executive voice: direct, commercial, data-driven, action-oriented.

═══════════════════════════════════════════════════════════════════
EXECUTIVE SUMMARY (80–150 words total across 4 parts):
Headline insight — 1 sentence capturing the dominant story in the data:

Supporting observation 1 — what the data shows and why it matters:

Supporting observation 2 — a second dimension with business implication:

Business implication — what leadership must consider or act on:

═══════════════════════════════════════════════════════════════════
BUSINESS HEALTH RATIONALE (2 sentences — overall condition + forward indicator):

═══════════════════════════════════════════════════════════════════
KEY FINDINGS (generate 6 findings — observation + explanation + business implication):
F1:
F2:
F3:
F4:
F5:
F6:

═══════════════════════════════════════════════════════════════════
TREND ANALYSIS (generate 5 trends — what happened + why it matters + what to watch):
T1:
T2:
T3:
T4:
T5:

═══════════════════════════════════════════════════════════════════
RISK ASSESSMENT (generate 5 risks — risk + likely impact + recommended mitigation):
R1:
R2:
R3:
R4:
R5:

═══════════════════════════════════════════════════════════════════
OPPORTUNITIES (generate 5 opportunities — opportunity + expected benefit + recommended action):
O1:
O2:
O3:
O4:
O5:

═══════════════════════════════════════════════════════════════════
STRATEGIC PRIORITIES (generate 3 priorities — each with title, rationale, expected outcome):
P1 Title:
P1 Rationale (why this must happen now, referencing the data):
P1 Expected Outcome (what success looks like in 90 days):

P2 Title:
P2 Rationale:
P2 Expected Outcome:

P3 Title:
P3 Rationale:
P3 Expected Outcome:

═══════════════════════════════════════════════════════════════════
EXECUTIVE TAKEAWAY (1 sentence, 20–30 words — the single most important message):

DATA:
---
{data_summary}
---"""


STEP2_SYSTEM = """You are a JSON formatter. Convert the input briefing into valid JSON.
Output ONLY the JSON object. No markdown fences. No preamble. No explanation."""

STEP2_USER = """Convert the executive briefing below into this EXACT JSON schema.

SCHEMA:
{{
  "executive_summary": "string — 80 to 150 words covering: headline insight, 2 supporting observations, 1 business implication. Full paragraph, not bullets.",
  "business_health_rationale": "string — 2 sentences: overall condition + forward indicator.",
  "key_findings": [
    "string — 25-40 words: observation + explanation + business implication",
    "string — 25-40 words: observation + explanation + business implication",
    "string — 25-40 words: observation + explanation + business implication",
    "string — 25-40 words: observation + explanation + business implication",
    "string — 25-40 words: observation + explanation + business implication",
    "string — 25-40 words: observation + explanation + business implication"
  ],
  "trends": [
    "string — 25-40 words: what happened + why it matters + what leadership should watch",
    "string — 25-40 words",
    "string — 25-40 words",
    "string — 25-40 words",
    "string — 25-40 words"
  ],
  "risks": [
    "string — 25-40 words: risk + likely impact + recommended mitigation",
    "string — 25-40 words",
    "string — 25-40 words",
    "string — 25-40 words",
    "string — 25-40 words"
  ],
  "opportunities": [
    "string — 25-40 words: opportunity + expected benefit + recommended action",
    "string — 25-40 words",
    "string — 25-40 words",
    "string — 25-40 words",
    "string — 25-40 words"
  ],
  "strategic_priorities": [
    {{
      "title": "2-5 word action label",
      "rationale": "string — 25-35 words explaining why this is the priority now",
      "outcome": "string — 15-25 words describing measurable success in 90 days"
    }},
    {{
      "title": "2-5 word action label",
      "rationale": "string — 25-35 words",
      "outcome": "string — 15-25 words"
    }},
    {{
      "title": "2-5 word action label",
      "rationale": "string — 25-35 words",
      "outcome": "string — 15-25 words"
    }}
  ],
  "executive_takeaway": "string — 20-30 words: single most important message for leadership"
}}

CRITICAL RULES:
- All 8 top-level keys must be present.
- key_findings, trends, risks, opportunities must each have EXACTLY 6, 5, 5, 5 items.
- strategic_priorities must have EXACTLY 3 objects, each with title + rationale + outcome.
- No item may be a single short label — every string must be a complete thought.
- No placeholder text ("N/A", "Not specified", "TBD", etc.).
- Output ONLY the JSON object. Start with {{ end with }}.

BRIEFING:
{briefing}"""


REPAIR_SYSTEM = "You are a JSON repair tool. Output ONLY valid JSON matching the schema. No other text."
REPAIR_USER = """Fix this JSON to match the schema exactly.

Required keys and types:
- executive_summary: string (80-150 words)
- business_health_rationale: string (2 sentences)
- key_findings: list of exactly 6 strings (25-40 words each)
- trends: list of exactly 5 strings (25-40 words each)
- risks: list of exactly 5 strings (25-40 words each)
- opportunities: list of exactly 5 strings (25-40 words each)
- strategic_priorities: list of exactly 3 objects, each with keys: title (str), rationale (str), outcome (str)
- executive_takeaway: string (20-30 words)

Rules:
- No item may be empty or a short label
- No placeholder text
- Output ONLY the fixed JSON

Broken JSON:
{broken}"""


# ════════════════════════════════════════════════════════════════════════════════
# SCHEMA VALIDATION
# ════════════════════════════════════════════════════════════════════════════════
SCHEMA = {
    "executive_summary":          str,
    "business_health_rationale":  str,
    "key_findings":               list,
    "trends":                     list,
    "risks":                      list,
    "opportunities":              list,
    "strategic_priorities":       list,
    "executive_takeaway":         str,
}

def validate(data: dict) -> tuple:
    for k, t in SCHEMA.items():
        if k not in data:
            return False, f"Missing key: {k}"
        if not isinstance(data[k], t):
            return False, f"Wrong type for {k}: expected {t.__name__}, got {type(data[k]).__name__}"
        if t == list and len(data[k]) == 0:
            return False, f"Empty list: {k}"
    # Validate strategic_priorities structure
    for i, p in enumerate(data.get("strategic_priorities", [])):
        if not isinstance(p, dict):
            return False, f"strategic_priorities[{i}] must be a dict"
        for key in ("title", "rationale", "outcome"):
            if key not in p or not str(p[key]).strip():
                return False, f"strategic_priorities[{i}] missing or empty '{key}'"
    return True, ""


# ════════════════════════════════════════════════════════════════════════════════
# ROBUST JSON EXTRACTOR (3 strategies)
# ════════════════════════════════════════════════════════════════════════════════
def extract_json(text: str) -> dict | None:
    text = text.strip()
    try: return json.loads(text)
    except: pass
    cleaned = re.sub(r"```(?:json)?\s*", "", text).replace("```", "").strip()
    try: return json.loads(cleaned)
    except: pass
    m = re.search(r"\{[\s\S]*\}", cleaned)
    if m:
        try: return json.loads(m.group())
        except: pass
    return None


# ════════════════════════════════════════════════════════════════════════════════
# LLM CALL + 2-STEP PIPELINE
# ════════════════════════════════════════════════════════════════════════════════
def llm(system: str, user: str, tag: str = "") -> str:
    print(f"  [{tag}] calling model...", end=" ", flush=True)
    t0 = time.time()
    r  = client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    print(f"{time.time()-t0:.1f}s")
    return r.choices[0].message.content.strip()


def generate_insights(data_summary: str) -> dict:
    print("\n── Step 1: Generating rich executive briefing ──")
    briefing = llm(STEP1_SYSTEM, STEP1_USER.format(data_summary=data_summary), "briefing")

    print("── Step 2: Converting to JSON schema ──")
    raw_json = llm(STEP2_SYSTEM, STEP2_USER.format(briefing=briefing), "json")
    insights = extract_json(raw_json)

    for attempt in range(MAX_RETRIES):
        if insights is not None:
            ok, err = validate(insights)
            if ok:
                break
            print(f"  ⚠️  Validation failed ({err}) — repair attempt {attempt+1}")
        else:
            print(f"  ⚠️  JSON extraction failed — repair attempt {attempt+1}")
        broken   = json.dumps(insights) if insights else raw_json
        repaired = llm(REPAIR_SYSTEM, REPAIR_USER.format(broken=broken[:4000]), "repair")
        insights = extract_json(repaired)

    if insights is None:
        raise ValueError("All JSON extraction attempts failed.")
    ok, err = validate(insights)
    if not ok:
        raise ValueError(f"Schema invalid after repair: {err}")

    # Health score calculated programmatically in build_deck
    insights["business_health_score"] = 50
    return insights


# ── Run ───────────────────────────────────────────────────────────────────────
if "data_summary" not in dir():
    print("⚠️  Run Cell 5 first.")
else:
    try:
        insights = generate_insights(data_summary)
        print("\n✅ Insights validated!")
        print(f"   Summary word count: {len(insights['executive_summary'].split())} words")
        print(f"   Findings: {len(insights['key_findings'])} items")
        print(f"   Trends: {len(insights['trends'])} items")
        print(f"   Risks: {len(insights['risks'])} items")
        print(f"   Opportunities: {len(insights['opportunities'])} items")
        print(f"   Priorities: {len(insights['strategic_priorities'])} items")
        print(f"   Takeaway: {insights['executive_takeaway'][:120]}")
    except Exception as e:
        print(f"\n❌ {e}")
        insights = None



── Step 1: Generating rich executive briefing ──
  [briefing] calling model... 8.5s
── Step 2: Converting to JSON schema ──
  [json] calling model... 6.0s

✅ Insights validated!
   Summary word count: 89 words
   Findings: 6 items
   Trends: 5 items
   Risks: 5 items
   Opportunities: 5 items
   Priorities: 3 items
   Takeaway: Addressing regional disparities in customer satisfaction and balancing our product portfolio are critical to sustaining 


In [9]:
# ════════════════════════════════════════════════════════════════════════════════
# LOW-LEVEL HELPERS
# ════════════════════════════════════════════════════════════════════════════════

MAX_PER_SIDE = 4

# Placeholder phrases the LLM sometimes emits — replace with professional fallbacks
_PLACEHOLDER_PHRASES = {
    "not specified", "not available", "no data", "n/a", "none",
    "not provided", "not mentioned", "not found", "not applicable",
    "not stated", "not included", "not discussed",
}

def _is_placeholder(text: str) -> bool:
    """Return True if text is a known placeholder or meaningless string."""
    t = str(text).strip().lower().rstrip(".")
    return (
        not t
        or t in _PLACEHOLDER_PHRASES
        or any(p in t for p in _PLACEHOLDER_PHRASES)
    )

def _safe_str(val, fallback: str = "") -> str:
    """Return a clean string, never None or 'None'."""
    if val is None:
        return fallback
    s = str(val).strip()
    return fallback if _is_placeholder(s) else s

def _sb(text) -> str:
    """Safe bullet: strip only. NO character truncation — let the text box wrap."""
    t = str(text).strip() if text is not None else ""
    return t if not _is_placeholder(t) else ""

# Professional fallback content per insight category
_FALLBACKS = {
    "key_findings": [
        "Dataset reviewed; quantitative summary is available — deeper segmentation is recommended to isolate performance drivers.",
        "Baseline metrics have been established, providing a foundation for comparative benchmarking against industry peers.",
        "Further data collection across additional periods would increase confidence in directional conclusions drawn here.",
        "Performance variance across segments suggests structural differences that merit focused leadership attention.",
        "Current data coverage is sufficient for directional insight but insufficient for precise forecasting without augmentation.",
        "Stakeholder alignment on these findings is a prerequisite for coordinated execution of any strategic response.",
    ],
    "trends": [
        "Trend direction is inconclusive from current dataset size; extending the observation period is strongly recommended before committing resources.",
        "Leading indicators suggest monitoring cadence should increase — early signals warrant closer tracking over the next quarter.",
        "Seasonal or cyclical patterns may be distorting observed trends; controlling for these effects will sharpen strategic interpretation.",
        "Peer benchmarking data would clarify whether observed trends reflect internal performance or broader market dynamics.",
        "The pace of change in key metrics suggests the business is at an inflection point requiring proactive management response.",
    ],
    "risks": [
        "Data coverage gaps limit confidence in risk quantification — investing in data completeness will improve forward decision quality.",
        "External market volatility remains an unquantified exposure; scenario planning against downside cases is recommended.",
        "Operational dependencies not fully visible in current dataset could mask concentration risks requiring urgent management attention.",
        "Reliance on a limited number of performance drivers creates fragility — diversification of revenue or operational inputs should be reviewed.",
        "Absence of leading-indicator monitoring means emerging risks may not surface until they have already affected results.",
    ],
    "opportunities": [
        "Efficiency gains are likely accessible through process optimisation — an operational review could unlock 10–15% cost improvement.",
        "Expanded data collection would unlock deeper segmentation value, enabling more precise targeting of high-return initiatives.",
        "Benchmarking against sector peers could reveal performance upside in areas where the business trails best-in-class operators.",
        "Untapped customer or market segments identified in the data represent a near-term revenue opportunity with manageable execution risk.",
        "Technology investment in analytics infrastructure would accelerate the time-to-insight cycle and improve strategic agility.",
    ],
}

def _sl(lst, mx=6, fallbacks=None) -> list:
    """
    Safe list: clean, deduplicate, cap at mx.
    fallbacks: professional strings used when result is empty.
    """
    if not lst or not isinstance(lst, list):
        lst = []
    items = [_sb(x) for x in lst if x is not None]
    items = [x for x in items if x]
    items = list(dict.fromkeys(items))   # deduplicate, preserve order
    items = items[:mx]
    if not items:
        items = fallbacks or [
            "Insufficient data to surface findings — review source dataset.",
            "Upload additional records to enable deeper analysis.",
        ]
    return items

def _ensure_list(data: dict, key: str, min_items: int = 2) -> list:
    """
    Return a clean, non-empty list for 'key' from insights dict.
    Injects professional fallback content when missing or placeholder-heavy.
    """
    raw = data.get(key) if isinstance(data, dict) else None
    if not isinstance(raw, list):
        raw = []
    cleaned = _sl(raw, mx=6, fallbacks=_FALLBACKS.get(key, [
        "Consult the full briefing for details on this section.",
    ]))
    if len(cleaned) < min_items:
        extras = [f for f in _FALLBACKS.get(key, []) if f not in cleaned]
        cleaned.extend(extras[:min_items - len(cleaned)])
    return cleaned

def _summarise_for_slide(text: str, max_sentences: int = 3) -> str:
    """
    Trim a multi-sentence string to at most max_sentences complete sentences.
    Never cuts mid-sentence. Falls back to the full text if splitting fails.
    """
    if not text or not isinstance(text, str):
        return "Refer to the full analysis for situation details."
    text = text.strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    if len(sentences) <= max_sentences:
        return text
    return " ".join(sentences[:max_sentences])

def _bg(slide, color):
    f = slide.background.fill; f.solid(); f.fore_color.rgb = color

def _rect(slide, l, t, w, h, color):
    s = slide.shapes.add_shape(1, Inches(l), Inches(t), Inches(w), Inches(h))
    s.fill.solid(); s.fill.fore_color.rgb = color; s.line.fill.background()
    return s

def _oval(slide, l, t, w, h, color):
    s = slide.shapes.add_shape(9, Inches(l), Inches(t), Inches(w), Inches(h))
    s.fill.solid(); s.fill.fore_color.rgb = color; s.line.fill.background()
    return s

def _tb(slide, text, l, t, w, h,
        size=14, bold=False, color=None, align=PP_ALIGN.LEFT,
        italic=False, font="Calibri", auto_fit=True):
    if color is None: color = C_DARK_TEXT
    safe_text = str(text).strip() if text is not None else ""
    txb = slide.shapes.add_textbox(Inches(l), Inches(t), Inches(w), Inches(h))
    tf  = txb.text_frame
    tf.word_wrap = True
    # Shrink font to fit as a last-resort safety net
    if auto_fit:
        tf.auto_size = MSO_AUTO_SIZE.TEXT_TO_FIT_SHAPE
    p   = tf.paragraphs[0]; p.alignment = align
    run = p.add_run()
    run.text = safe_text
    run.font.size = Pt(size)
    run.font.bold = bold; run.font.italic = italic
    run.font.color.rgb = color; run.font.name = font
    return txb

def _bullets(slide, items, l, t, w, h, size=14, color=None, marker="▸"):
    if color is None: color = C_DARK_TEXT
    # Sanitise: remove None, empty, and placeholder strings
    clean_items = [_safe_str(x) for x in (items or []) if _safe_str(x)]
    if not clean_items:
        clean_items = ["No data available for this section."]
    txb = slide.shapes.add_textbox(Inches(l), Inches(t), Inches(w), Inches(h))
    tf  = txb.text_frame
    tf.word_wrap = True
    tf.auto_size = MSO_AUTO_SIZE.TEXT_TO_FIT_SHAPE
    for i, item in enumerate(clean_items):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.space_before = Pt(4)
        p.alignment = PP_ALIGN.LEFT
        run = p.add_run()
        run.text = f"{marker}  {item}"
        run.font.size = Pt(size); run.font.color.rgb = color; run.font.name = "Calibri"
    return txb

def _add_image(slide, img_path, l, t, w, h):
    """Add a PNG chart image."""
    if img_path and Path(img_path).exists():
        slide.shapes.add_picture(img_path, Inches(l), Inches(t),
                                 Inches(w), Inches(h))
        return True
    return False


# ════════════════════════════════════════════════════════════════════════════════
# HEALTH SCORE GAUGE  (drawn with shapes — no images needed)
# ════════════════════════════════════════════════════════════════════════════════
def _health_gauge(slide, score: int, cx: float, cy: float):
    """
    Draw a simple circular gauge using nested ovals.
    cx, cy = centre in inches.
    """
    r_outer = 1.1
    _oval(slide, cx - r_outer, cy - r_outer, r_outer*2, r_outer*2, C_OFFWHITE)

    if score >= 70:   arc_col = C_GREEN
    elif score >= 40: arc_col = C_GOLD
    else:             arc_col = C_RED_SOFT

    r_mid = 0.85
    _oval(slide, cx - r_mid, cy - r_mid, r_mid*2, r_mid*2, arc_col)

    r_inner = 0.62
    _oval(slide, cx - r_inner, cy - r_inner, r_inner*2, r_inner*2, C_WHITE)

    _tb(slide, str(score), cx - 0.5, cy - 0.38, 1.0, 0.6,
        size=28, bold=True, color=arc_col, align=PP_ALIGN.CENTER)
    _tb(slide, "/100", cx - 0.45, cy + 0.05, 0.9, 0.3,
        size=11, color=C_MUTED, align=PP_ALIGN.CENTER)


# ════════════════════════════════════════════════════════════════════════════════
# KPI CARD — arrow auto-calculated from sentiment
# ════════════════════════════════════════════════════════════════════════════════
def _kpi_card(slide, label, value, delta, sentiment, l, t, w=2.9, h=1.6):
    """Draw a KPI metric card with auto-calculated direction arrow."""
    # Sanitise inputs
    label     = str(label or "Metric").strip()
    value     = str(value or "—").strip()
    delta_raw = str(delta or "").strip()
    sentiment = str(sentiment or "neutral").strip().lower()

    # Strip any existing arrow so we control it ourselves
    delta_clean = delta_raw.lstrip("▲▼—–- ").strip()

    # Auto-calculate arrow from sentiment (deterministic)
    if sentiment == "positive":
        arrow = "▲"; delta_color = C_GREEN
    elif sentiment == "negative":
        arrow = "▼"; delta_color = C_RED_SOFT
    else:
        arrow = "—"; delta_color = C_MUTED

    delta_display = f"{arrow}  {delta_clean}" if delta_clean else arrow

    _rect(slide, l, t, w, h, C_WHITE)
    band_col = C_TEAL if sentiment == "positive" else (C_RED_SOFT if sentiment == "negative" else C_GOLD)
    _rect(slide, l, t, w, 0.07, band_col)
    _tb(slide, label,         l+0.15, t+0.12, w-0.2, 0.35, size=10, color=C_MUTED)
    _tb(slide, value,         l+0.12, t+0.42, w-0.2, 0.62, size=26, bold=True, color=C_DARK_TEXT)
    _tb(slide, delta_display, l+0.12, t+1.15, w-0.2, 0.30, size=11, bold=True, color=delta_color)


# ════════════════════════════════════════════════════════════════════════════════
# SECTION DIVIDER SLIDE
# ════════════════════════════════════════════════════════════════════════════════
def _section_divider(prs, number: str, title: str, subtitle: str = ""):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_NAVY)

    _tb(slide, number, 8.5, 0.8, 4.5, 5.5,
        size=200, bold=True, color=RGBColor(0x17, 0x2A, 0x6A),
        align=PP_ALIGN.RIGHT, font="Cambria")

    _oval(slide, 0.7, 3.1, 0.18, 0.18, C_TEAL)

    _tb(slide, title, 1.1, 2.7, 9.0, 1.2,
        size=40, bold=True, color=C_WHITE, font="Cambria")
    if subtitle:
        _tb(slide, subtitle, 1.1, 4.0, 9.0, 0.6,
            size=16, italic=True, color=C_TEAL)
    return slide


# ════════════════════════════════════════════════════════════════════════════════
# SLIDE BUILDERS
# ════════════════════════════════════════════════════════════════════════════════

def slide_01_cover(prs, filename, ts):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_CHARCOAL)

    _rect(slide, 10.2, 0, 3.13, 7.5, RGBColor(0x00, 0x9E, 0x92))
    _rect(slide, 10.2, 0, 0.25, 7.5, RGBColor(0x00, 0x7A, 0x72))

    for row in range(5):
        for col in range(4):
            _oval(slide, 10.55 + col*0.62, 1.0 + row*1.1, 0.1, 0.1,
                  RGBColor(0xFF, 0xFF, 0xFF))

    _tb(slide, "EXECUTIVE INTELLIGENCE PLATFORM", 0.8, 1.5, 9.0, 0.5,
        size=11, bold=True, color=C_TEAL, font="Calibri")
    _tb(slide, "Executive\nInsight Report", 0.8, 2.1, 9.0, 2.4,
        size=48, bold=True, color=C_WHITE, font="Cambria")
    _tb(slide, "AI-Powered Business Intelligence · Strategic Analysis · Executive Briefing",
        0.8, 4.6, 9.0, 0.5, size=13, italic=True, color=C_MUTED)

    _rect(slide, 0.8, 5.5, 4.5, 0.03, C_TEAL)
    _tb(slide, f"Source: {filename}", 0.8, 5.6, 5.0, 0.35, size=11, color=C_MUTED)
    _tb(slide, ts, 0.8, 5.95, 5.0, 0.35, size=11, color=C_MUTED)
    _tb(slide, "Qwen2.5-7B-Instruct via vLLM", 0.8, 6.3, 5.0, 0.35,
        size=10, italic=True, color=RGBColor(0x44, 0x44, 0x66))


def slide_03_exec_summary(prs, summary: str, score: int, rationale: str, takeaway: str):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_OFFWHITE)

    # Header
    _rect(slide, 0, 0, 13.33, 1.05, C_CHARCOAL)
    _tb(slide, "Executive Summary", 0.55, 0.15, 9.0, 0.78,
        size=28, bold=True, color=C_WHITE, font="Cambria")
    _tb(slide, "Situation Overview", 9.6, 0.28, 3.3, 0.5,
        size=11, color=C_TEAL, align=PP_ALIGN.RIGHT)

    # Summary text — left 2/3 (reduced height to leave gap before takeaway banner)
    _rect(slide, 0.5, 1.25, 8.6, 4.1, C_WHITE)
    _tb(slide, "THE SITUATION", 0.75, 1.4, 4.0, 0.35, size=9, bold=True, color=C_TEAL)
    # Executive summary is now a full paragraph (80-150 words) — render in full
    display_summary = _safe_str(summary, "Data analysis complete. Review appendix for full detail.")
    _tb(slide, display_summary, 0.75, 1.75, 8.15, 3.45, size=14, color=C_DARK_TEXT)

    # Business Health Score — right 1/3 (gauge)
    _rect(slide, 9.4, 1.25, 3.4, 4.1, C_WHITE)
    _tb(slide, "BUSINESS HEALTH", 9.55, 1.42, 3.1, 0.35,
        size=9, bold=True, color=C_TEAL, align=PP_ALIGN.CENTER)
    _health_gauge(slide, score, cx=11.1, cy=3.05)
    _tb(slide, _summarise_for_slide(_safe_str(rationale, "Score reflects available KPI signals."), max_sentences=2),
        9.55, 4.25, 3.1, 0.9,
        size=10, italic=True, color=C_MUTED, align=PP_ALIGN.CENTER)

    # Takeaway banner — fixed at bottom with clear gap (content box ends at y=5.35)
    _rect(slide, 0.5, 5.5, 12.33, 1.5, C_NAVY)
    _tb(slide, "KEY TAKEAWAY", 0.8, 5.6, 2.5, 0.3, size=9, bold=True, color=C_TEAL)
    _tb(slide, _safe_str(takeaway, "Review full briefing for the executive takeaway."),
        0.8, 5.92, 11.8, 0.85, size=13, bold=True, color=C_WHITE, italic=True)


def slide_04_kpi_dashboard(prs, kpis: list):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_OFFWHITE)
    _rect(slide, 0, 0, 13.33, 1.05, C_CHARCOAL)
    _tb(slide, "KPI Dashboard", 0.55, 0.15, 8.0, 0.78,
        size=28, bold=True, color=C_WHITE, font="Cambria")
    _tb(slide, "Key Performance Indicators", 9.0, 0.28, 3.9, 0.5,
        size=11, color=C_TEAL, align=PP_ALIGN.RIGHT)

    if not kpis:
        _tb(slide, "KPI data not available for this file type.",
            1.0, 2.5, 11.0, 1.0, size=16, color=C_MUTED, align=PP_ALIGN.CENTER)
        return

    cols  = 3
    xs    = [0.55, 4.75, 8.95]
    ys    = [1.25, 3.15]
    for idx, kpi in enumerate(kpis[:6]):
        c = idx % cols
        r = idx // cols
        _kpi_card(slide, kpi.get("label",""), kpi.get("value",""), kpi.get("delta",""),
                  kpi.get("sentiment","neutral"), xs[c], ys[r])

    _tb(slide, "▲ positive trend   ▼ negative trend   — stable",
        0.55, 6.9, 12.0, 0.4, size=9, color=C_MUTED, italic=True)


def slide_findings(prs, heading, items, icon_char, accent_color, slide_num, total):
    """Generic findings slide with smart 1/2-column layout + numbered icon."""
    items = _sl(items, mx=7, fallbacks=_FALLBACKS.get(heading.lower().replace(" ","_"), None))
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_OFFWHITE)

    # Header
    _rect(slide, 0, 0, 13.33, 1.05, C_CHARCOAL)
    _tb(slide, heading, 0.55, 0.15, 9.5, 0.78,
        size=28, bold=True, color=C_WHITE, font="Cambria")
    _tb(slide, f"{slide_num} of {total}", 10.5, 0.3, 2.4, 0.45,
        size=11, color=C_MUTED, align=PP_ALIGN.RIGHT)

    # Icon circle — y=1.2 to leave clear gap before content box
    _oval(slide, 0.55, 1.2, 0.75, 0.75, accent_color)
    _tb(slide, icon_char, 0.55, 1.25, 0.75, 0.62,
        size=18, bold=True, color=C_WHITE, align=PP_ALIGN.CENTER)

    # Content boxes — start at y=2.2, max height 4.9 (ends at y=7.1, within 7.5 slide)
    if len(items) <= 3:
        _rect(slide, 0.55, 2.2, 12.25, 4.9, C_WHITE)
        _bullets(slide, items, 0.9, 2.4, 11.8, 4.5, size=15, marker="▸")
    else:
        mid  = (len(items)+1)//2
        col1, col2 = items[:mid], items[mid:]
        _rect(slide, 0.55, 2.2, 5.9, 4.9, C_WHITE)
        _bullets(slide, col1, 0.85, 2.4, 5.5, 4.5, size=14)
        _rect(slide, 6.9, 2.2, 5.9, 4.9, C_WHITE)
        _bullets(slide, col2, 7.2, 2.4, 5.5, 4.5, size=14)


def slide_trend_with_chart(prs, trends, chart_path):
    """Trends slide: text on left, chart on right."""
    items = _sl(trends, 4, fallbacks=_FALLBACKS["trends"])
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_OFFWHITE)
    _rect(slide, 0, 0, 13.33, 1.05, C_CHARCOAL)
    _tb(slide, "Trend Analysis", 0.55, 0.15, 7.0, 0.78,
        size=28, bold=True, color=C_WHITE, font="Cambria")
    _tb(slide, "Direction & Momentum", 9.5, 0.28, 3.4, 0.5,
        size=11, color=C_TEAL, align=PP_ALIGN.RIGHT)

    _oval(slide, 0.55, 1.25, 0.7, 0.7, C_TEAL)
    _tb(slide, "📈", 0.55, 1.3, 0.7, 0.58,
        size=18, bold=True, color=C_WHITE, align=PP_ALIGN.CENTER)
    _rect(slide, 0.55, 2.15, 6.1, 5.0, C_WHITE)
    _bullets(slide, items, 0.85, 2.35, 5.7, 4.6, size=14)

    if chart_path and Path(chart_path).exists():
        slide.shapes.add_picture(chart_path, Inches(6.95), Inches(1.25),
                                 Inches(5.9), Inches(5.9))
    else:
        _rect(slide, 6.95, 1.25, 5.9, 5.9, C_WHITE)
        _tb(slide, "Chart not available", 7.2, 3.5, 5.3, 0.5,
            size=13, color=C_MUTED, align=PP_ALIGN.CENTER, italic=True)


def slide_priorities(prs, priorities):
    """
    Strategic priorities slide.
    Accepts either:
      - list of dicts: [{"title": str, "rationale": str, "outcome": str}, ...]
      - list of strings (legacy fallback)
    Renders 3 cards with title banner + rationale + outcome label.
    """
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_OFFWHITE)
    _rect(slide, 0, 0, 13.33, 1.05, C_CHARCOAL)
    _tb(slide, "Strategic Priorities", 0.55, 0.15, 9.0, 0.78,
        size=28, bold=True, color=C_WHITE, font="Cambria")
    _tb(slide, "Leadership Action Plan", 9.5, 0.28, 3.4, 0.5,
        size=11, color=C_GOLD, align=PP_ALIGN.RIGHT)

    card_colors = [C_TEAL, C_GOLD, C_GREEN]
    xs = [0.55, 4.88, 9.2]

    # Normalise to list of dicts
    normalised = []
    for item in (priorities or [])[:3]:
        if isinstance(item, dict):
            normalised.append({
                "title":     _safe_str(item.get("title"),    f"Priority {len(normalised)+1}"),
                "rationale": _safe_str(item.get("rationale"), "Rationale to be confirmed."),
                "outcome":   _safe_str(item.get("outcome"),  "Outcome to be confirmed."),
            })
        else:
            # Legacy string format — split into title and body
            text = _safe_str(item, f"Priority {len(normalised)+1}")
            parts = text.split(".", 1)
            normalised.append({
                "title":     parts[0].strip()[:60],
                "rationale": parts[1].strip() if len(parts) > 1 else text,
                "outcome":   "Measure progress at next leadership review.",
            })

    # Pad with fallbacks if fewer than 3
    fallback_priorities = [
        {"title": "Establish Measurement Framework",
         "rationale": "Reliable KPI tracking is the foundation for all future data-driven decisions and must be prioritised immediately.",
         "outcome": "All key metrics tracked weekly within 60 days."},
        {"title": "Invest in Data Quality",
         "rationale": "Current data gaps reduce confidence in strategic decisions; improving coverage will unlock deeper analytical value.",
         "outcome": "Data completeness above 90% within 90 days."},
        {"title": "Initiate Stakeholder Review",
         "rationale": "Leadership alignment on findings ensures coordinated action and avoids fragmented responses to shared challenges.",
         "outcome": "Cross-functional action plan agreed within 30 days."},
    ]
    while len(normalised) < 3:
        normalised.append(fallback_priorities[len(normalised)])

    for i, (p, x, col) in enumerate(zip(normalised, xs, card_colors)):
        # Card shell
        _rect(slide, x, 1.3, 3.6, 5.8, C_WHITE)
        # Colour banner top
        _rect(slide, x, 1.3, 3.6, 1.1, col)
        # Priority number
        _tb(slide, f"0{i+1}", x+0.12, 1.32, 0.95, 1.05,
            size=38, bold=True, color=C_WHITE, font="Cambria", align=PP_ALIGN.LEFT)
        # Title on banner
        _tb(slide, p["title"].upper(), x+1.05, 1.38, 2.4, 1.0,
            size=11, bold=True, color=C_WHITE, align=PP_ALIGN.LEFT)

        # Rationale label
        _tb(slide, "WHY NOW", x+0.18, 2.55, 3.0, 0.28,
            size=8, bold=True, color=col)
        # Rationale body
        rationale_len = len(p["rationale"])
        rat_size = 12 if rationale_len <= 130 else (11 if rationale_len <= 200 else 10)
        _tb(slide, p["rationale"], x+0.18, 2.85, 3.25, 2.1,
            size=rat_size, color=C_DARK_TEXT, auto_fit=True)

        # Outcome label
        _tb(slide, "EXPECTED OUTCOME", x+0.18, 5.05, 3.0, 0.28,
            size=8, bold=True, color=col)
        # Outcome body
        out_size = 11 if len(p["outcome"]) <= 100 else 10
        _tb(slide, p["outcome"], x+0.18, 5.35, 3.25, 0.6,
            size=out_size, italic=True, color=C_MUTED, auto_fit=True)


def slide_takeaway(prs, takeaway: str, summary: str):
    """Bold single-message takeaway slide."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_CHARCOAL)

    _tb(slide, "\u201C", 0.5, 0.2, 2.5, 2.5,
        size=160, color=RGBColor(0x2A, 0x2A, 0x4A), bold=True, font="Cambria")

    _tb(slide, "THE BOTTOM LINE", 1.0, 1.0, 11.0, 0.45, size=11, bold=True, color=C_TEAL)
    _tb(slide, _safe_str(takeaway, "Act on the findings — the window for intervention is now."),
        1.0, 1.55, 11.3, 2.5, size=26, bold=True, color=C_WHITE, font="Cambria")
    _tb(slide, _summarise_for_slide(_safe_str(summary, "Performance data reviewed; strategic response recommended."), 2),
        1.0, 4.4, 11.3, 1.8, size=14, italic=True, color=C_MUTED)

    _rect(slide, 1.0, 6.5, 6.0, 0.04, C_TEAL)


def slide_appendix_chart(prs, chart_path: str, title: str):
    """Full-slide chart for appendix."""
    if not chart_path or not Path(chart_path).exists():
        return
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_OFFWHITE)
    _rect(slide, 0, 0, 13.33, 1.05, C_CHARCOAL)
    _tb(slide, title, 0.55, 0.15, 12.3, 0.78,
        size=22, bold=True, color=C_WHITE, font="Cambria")
    slide.shapes.add_picture(chart_path, Inches(0.6), Inches(1.2),
                             Inches(12.1), Inches(5.9))


def slide_close(prs, filename):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _bg(slide, C_CHARCOAL)
    _rect(slide, 10.5, 0, 2.83, 7.5, RGBColor(0x00, 0x9E, 0x92))

    _tb(slide, "Thank You", 1.0, 2.1, 9.0, 1.6,
        size=54, bold=True, color=C_WHITE, font="Cambria")
    _tb(slide, "This report was generated automatically using AI.",
        1.0, 3.9, 9.0, 0.6, size=14, italic=True, color=C_MUTED)
    _tb(slide, f"Source: {filename}",
        1.0, 4.6, 9.0, 0.4, size=12, color=C_MUTED)
    _tb(slide, "Powered by Qwen2.5-7B-Instruct · vLLM · python-pptx",
        1.0, 5.05, 9.0, 0.4, size=11, italic=True,
        color=RGBColor(0x44, 0x44, 0x66))


# ════════════════════════════════════════════════════════════════════════════════
# MASTER BUILD FUNCTION
# ════════════════════════════════════════════════════════════════════════════════
def build_deck(insights: dict, kpis: list, chart_paths: dict,
               filename: str, out_path: str) -> str:
    prs = Presentation()
    prs.slide_width  = SLIDE_W
    prs.slide_height = SLIDE_H

    ts = datetime.now().strftime("%B %d, %Y")

    # Recalculate health score from final kpis — deterministic, never LLM-generated
    insights["business_health_score"] = calculate_health_score(kpis, insights)

    # ── Pre-calculate total slide count for accurate footer numbering ───────────
    # Base: cover(1)+divider(1)+exec(1)+kpi(1)+findings(1)+trends(1)+risks(1)
    #       +opps(1)+divider(1)+priorities(1)+takeaway(1)+close(1) = 12
    total_slides = 12
    if chart_paths.get("category"):     total_slides += 1   # appendix A
    if chart_paths.get("distribution"): total_slides += 1   # appendix B

    # Content slides that show "X of Y" are: findings, risks, opportunities
    content_total = 3

    # ── PART 1: THE SITUATION ──────────────────────────────────────────────────
    slide_01_cover(prs, filename, ts)
    _section_divider(prs, "01", "The Situation", "What does the data tell us?")
    slide_03_exec_summary(
        prs,
        _safe_str(insights.get("executive_summary"),
                  "The data indicates mixed performance across key metrics."),
        insights.get("business_health_score", 50),
        _safe_str(insights.get("business_health_rationale"),
                  "Score reflects available KPI signals and trend data."),
        _safe_str(insights.get("executive_takeaway"),
                  "Leadership should prioritise data-driven decisions now."),
    )
    slide_04_kpi_dashboard(prs, kpis)
    slide_findings(prs, "Key Findings",
                   _ensure_list(insights, "key_findings"),
                   "🔍", C_TEAL, 1, content_total)
    slide_trend_with_chart(prs,
                           _ensure_list(insights, "trends"),
                           chart_paths.get("trend"))

    # ── PART 2: RISK & OPPORTUNITY ────────────────────────────────────────────
    slide_findings(prs, "Risk Assessment",
                   _ensure_list(insights, "risks"),
                   "⚠", C_RED_SOFT, 2, content_total)
    slide_findings(prs, "Opportunities",
                   _ensure_list(insights, "opportunities"),
                   "💡", C_GREEN, 3, content_total)

    # ── PART 3: THE PATH FORWARD ──────────────────────────────────────────────
    _section_divider(prs, "02", "The Path Forward", "What leadership must do next.")
    slide_priorities(prs, insights.get("strategic_priorities", []))
    slide_takeaway(prs,
                   _safe_str(insights.get("executive_takeaway"),
                             "Act on the findings — the window for intervention is now."),
                   _safe_str(insights.get("executive_summary"),
                             "Performance data reviewed; strategic response recommended."))

    # ── APPENDIX: CHARTS ──────────────────────────────────────────────────────
    if chart_paths.get("category"):
        slide_appendix_chart(prs, chart_paths["category"], "Appendix A — Category Breakdown")
    if chart_paths.get("distribution"):
        slide_appendix_chart(prs, chart_paths["distribution"], "Appendix B — Distribution Analysis")

    slide_close(prs, filename)

    prs.save(out_path)
    return out_path


# ── Run ───────────────────────────────────────────────────────────────────────
if "insights" not in dir() or insights is None:
    print("⚠️  Run Cell 7 (Insight Generation) first.")
else:
    fname = uploaded_data.get("filename", "data")
    out   = build_deck(
        insights,
        kpis         if "kpis" in dir() else [],
        chart_paths  if "chart_paths" in dir() else {},
        fname,
        OUTPUT_PPT
    )
    sz = Path(out).stat().st_size / 1024
    print(f"✅ Deck saved: {out}")
   


✅ Deck saved: Executive_Intelligence_Report.pptx


In [10]:
def make_link(filepath):
    p = Path(filepath)
    if not p.exists():
        print(f"❌ File not found: {filepath} — run Cell 8 first."); return
    b64  = base64.b64encode(p.read_bytes()).decode()
    size = p.stat().st_size / 1024
    ts   = datetime.now().strftime("%H:%M:%S")
    display(HTML(f"""
    <div style="background:linear-gradient(135deg,#1C1C2E 0%,#0D1B4B 100%);
                border-radius:14px;padding:28px 38px;display:inline-block;
                margin:12px 0;box-shadow:0 8px 24px rgba(0,0,0,.3)">
      <p style="color:#00C2B2;font-family:Calibri,Arial,sans-serif;
                font-size:11px;font-weight:700;letter-spacing:2px;
                text-transform:uppercase;margin:0 0 6px">Executive Intelligence Platform</p>
      <p style="color:#8899BB;font-family:Calibri,Arial,sans-serif;
                font-size:12px;margin:0 0 18px">
        ✅ Report ready &nbsp;·&nbsp; {size:.0f} KB &nbsp;·&nbsp; Generated at {ts}
      </p>
      <a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}"
         download="{p.name}"
         style="background:#00C2B2;color:#0D1B4B;padding:13px 32px;
                border-radius:8px;text-decoration:none;
                font-family:Calibri,Arial,sans-serif;font-size:16px;
                font-weight:900;letter-spacing:.5px">
        ⬇ &nbsp; Download {p.name}
      </a>
    </div>"""))

make_link(OUTPUT_PPT)